# 20｜不用 `nn.MultiheadAttention/nn.Transformer`：手写 Seq2Seq Transformer

本笔记从 token/位置嵌入、缩放点积注意力、多头拆并、padding/causal mask、编码器/解码器块一路实现到 `Seq2SeqTransformer.forward` 与贪心解码。目标不是复刻大型语言模型，而是把“未来信息隔离、teacher forcing、权重共享与制品合同”变成可执行测试。

## 1. 张量与特殊 token 合同

- token 张量：`[B,T]`，`PAD=0, BOS=1, EOS=2`，普通 token 从 3 开始。
- 隐状态：`[B,T,D]`；多头后：`[B,H,T,D/H]`，要求 `D % H == 0`。
- 注意力允许掩码是 bool，形状可广播到 `[B,H,T_q,T_k]`，`True` 表示允许读取。
- 训练输入 `tgt_in` 以 BOS 开头；监督 `tgt_out` 相对左移一位，以 EOS 结束。
- 本实验只在固定小集合上受控过拟合，不把结果解释为序列泛化。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

import hashlib
import io
import math
import random

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260729
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
PAD, BOS, EOS = 0, 1, 2

assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
assert len({PAD, BOS, EOS}) == 3
print({"torch": torch.__version__, "seed": SEED})

## 2. 缩放点积注意力

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+M\right)V.$$

对禁止位置，$M=-\infty$。工程上还必须处理“一整行都被屏蔽”：本合同保证每个 decoder 查询至少能看到 BOS，每个有效 encoder 查询至少能看到一个有效源 token。函数仍显式检查全屏蔽行，避免 softmax 产生 NaN。

In [ ]:
def scaled_dot_product_attention(q, k, v, allowed_mask=None):
    if q.shape[:-2] != k.shape[:-2] or k.shape != v.shape:
        raise ValueError("q/k/v 的 batch、head、通道合同不成立")
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    if allowed_mask is not None:
        if allowed_mask.dtype != torch.bool:
            raise TypeError("allowed_mask 必须是 bool")
        expanded = torch.broadcast_to(allowed_mask, scores.shape)
        if bool((~expanded.any(dim=-1)).any()):
            raise ValueError("存在完全不可见的 query")
        scores = scores.masked_fill(~expanded, torch.finfo(scores.dtype).min)
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights

q0 = torch.tensor([[[[1.0, 0.0]]]])
k0 = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])
v0 = torch.tensor([[[[3.0, 0.0], [0.0, 5.0]]]])
ctx0, w0 = scaled_dot_product_attention(q0, k0, v0, torch.tensor([[[[True, False]]]]))
assert ctx0.shape == (1, 1, 1, 2)
assert torch.allclose(ctx0, v0[:, :, :1])
assert torch.allclose(w0.sum(-1), torch.ones_like(w0.sum(-1)))

## 3. 多头拆分与合并

四个线性层分别产生 Q/K/V 与输出投影。`split_heads` 把 `[B,T,D]` 变为 `[B,H,T,d_h]`；`merge_heads` 做逆变换。我们没有调用任何预制注意力或 Transformer 层，因此每一步 shape 都可观察。

In [ ]:
class MultiHeadAttentionFromScratch(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model % num_heads:
            raise ValueError("d_model 必须能被 num_heads 整除")
        self.d_model, self.num_heads = d_model, num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

    def merge_heads(self, x):
        B, H, T, Dh = x.shape
        if H != self.num_heads or Dh != self.head_dim:
            raise ValueError("head shape 不符合配置")
        return x.transpose(1, 2).contiguous().view(B, T, self.d_model)

    def forward(self, query, key, value, allowed_mask=None, need_weights=False):
        q, k, v = map(self.split_heads, (self.q_proj(query), self.k_proj(key), self.v_proj(value)))
        context, weights = scaled_dot_product_attention(q, k, v, allowed_mask)
        out = self.o_proj(self.merge_heads(context))
        return (out, weights) if need_weights else out

mha0 = MultiHeadAttentionFromScratch(12, 3)
x0 = torch.randn(2, 5, 12)
out0, weights0 = mha0(x0, x0, x0, torch.ones(2, 1, 5, 5, dtype=torch.bool), True)
assert out0.shape == (2, 5, 12)
assert weights0.shape == (2, 3, 5, 5)
assert torch.allclose(weights0.sum(-1), torch.ones(2, 3, 5), atol=1e-6)

## 4. padding mask 与 causal mask 的组合

Encoder self-attention 屏蔽 padding key；decoder self-attention 同时要求 `key_position <= query_position` 且 key 非 padding；cross-attention 只屏蔽源 padding key。查询位置的 padding 最终会在 block 外清零，以免残差携带垃圾值。

注意：causal mask 的方向非常容易写反，必须用具体矩阵和“改未来不影响过去”的属性测试。

In [ ]:
def make_masks(src_tokens, tgt_tokens):
    if src_tokens.ndim != 2 or tgt_tokens.ndim != 2:
        raise ValueError("token 张量必须为 [B,T]")
    B, S = src_tokens.shape
    _, T = tgt_tokens.shape
    src_key = (src_tokens != PAD)[:, None, None, :]           # [B,1,1,S]
    tgt_key = (tgt_tokens != PAD)[:, None, None, :]           # [B,1,1,T]
    causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=tgt_tokens.device))[None, None]
    tgt_self = tgt_key & causal
    cross = src_key
    return src_key, tgt_self, cross

src_probe = torch.tensor([[4, 5, PAD], [6, 7, 8]])
tgt_probe = torch.tensor([[BOS, 9, PAD], [BOS, 8, 7]])
src_mask0, tgt_mask0, cross_mask0 = make_masks(src_probe, tgt_probe)
assert src_mask0.shape == (2, 1, 1, 3)
assert tgt_mask0.shape == (2, 1, 3, 3)
assert not tgt_mask0[0, 0, 0, 1]
assert tgt_mask0[0, 0, 1, 0]
assert not tgt_mask0[0, 0, 2, 2]
assert torch.equal(src_mask0, cross_mask0)

## 5. Token + position embedding

可学习位置表把顺序注入模型。token embedding 乘 $\sqrt{D}$，避免其初始尺度相对位置向量过小。超过 `max_len` 直接拒绝，不能静默截断，因为截断会改变监督对齐。

In [ ]:
class TokenPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len):
        super().__init__()
        self.token = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.position = nn.Embedding(max_len, d_model)
        self.d_model, self.max_len = d_model, max_len

    def forward(self, tokens):
        if tokens.shape[1] > self.max_len:
            raise ValueError("序列超过位置表长度")
        pos = torch.arange(tokens.shape[1], device=tokens.device)
        return self.token(tokens) * math.sqrt(self.d_model) + self.position(pos)[None]

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

class EncoderBlock(nn.Module):
    def __init__(self, d_model, heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttentionFromScratch(d_model, heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
    def forward(self, x, src_mask, query_valid):
        x = self.norm1(x + self.attn(x, x, x, src_mask))
        x = self.norm2(x + self.ffn(x))
        return x * query_valid.unsqueeze(-1)

class DecoderBlock(nn.Module):
    def __init__(self, d_model, heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttentionFromScratch(d_model, heads)
        self.cross_attn = MultiHeadAttentionFromScratch(d_model, heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1, self.norm2, self.norm3 = (nn.LayerNorm(d_model) for _ in range(3))
    def forward(self, x, memory, self_mask, cross_mask, query_valid):
        x = self.norm1(x + self.self_attn(x, x, x, self_mask))
        x = self.norm2(x + self.cross_attn(x, memory, memory, cross_mask))
        x = self.norm3(x + self.ffn(x))
        return x * query_valid.unsqueeze(-1)

emb0 = TokenPositionEmbedding(20, 12, 8)
assert emb0(torch.tensor([[BOS, 4, EOS]])).shape == (1, 3, 12)
assert emb0.token.padding_idx == PAD

## 6. Encoder、Decoder 与权重共享

模型先编码源序列，再以 causal self-attention 和 cross-attention 解码。输出投影与目标 token embedding 共享同一个 `Parameter`：这减少参数并要求词表与隐藏维度兼容。共享不能只靠数值相等验证，要检查对象/存储地址。

单个带 bias 的多头注意力参数量为 $4(D^2+D)$；两层 FFN 为 $2DD_{ff}+D_{ff}+D$。总参数还包括两个词嵌入/位置表、LayerNorm 与层数倍数。代码用 `model.parameters()` 对唯一 Parameter 求和，因此被绑定的输出矩阵不会重复计数。

In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=32, heads=4, d_ff=64, layers=1, max_len=16):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, d_model=d_model, heads=heads,
                           d_ff=d_ff, layers=layers, max_len=max_len)
        self.src_embed = TokenPositionEmbedding(vocab_size, d_model, max_len)
        self.tgt_embed = TokenPositionEmbedding(vocab_size, d_model, max_len)
        self.encoders = nn.ModuleList([EncoderBlock(d_model, heads, d_ff) for _ in range(layers)])
        self.decoders = nn.ModuleList([DecoderBlock(d_model, heads, d_ff) for _ in range(layers)])
        self.output = nn.Linear(d_model, vocab_size, bias=False)
        self.output.weight = self.tgt_embed.token.weight

    def encode(self, src):
        src_mask, _, _ = make_masks(src, torch.full((src.shape[0], 1), BOS, device=src.device))
        valid = src != PAD
        memory = self.src_embed(src) * valid.unsqueeze(-1)
        for block in self.encoders:
            memory = block(memory, src_mask, valid)
        return memory, src_mask

    def decode(self, tgt_in, memory, src_mask):
        _, self_mask, cross = make_masks(torch.ones(src_mask.shape[0], src_mask.shape[-1],
                                                    dtype=torch.long, device=tgt_in.device), tgt_in)
        cross = src_mask
        valid = tgt_in != PAD
        x = self.tgt_embed(tgt_in) * valid.unsqueeze(-1)
        for block in self.decoders:
            x = block(x, memory, self_mask, cross, valid)
        return self.output(x)

    def forward(self, src, tgt_in):
        memory, src_mask = self.encode(src)
        return self.decode(tgt_in, memory, src_mask)

model20 = Seq2SeqTransformer(vocab_size=14)
dummy_logits = model20(torch.tensor([[4, 5, EOS]]), torch.tensor([[BOS, 5, 4]]))
parameter_count20 = sum(p.numel() for p in model20.parameters())  # 共享 Parameter 只计一次
assert dummy_logits.shape == (1, 3, 14)
assert parameter_count20 > 10_000
assert model20.output.weight is model20.tgt_embed.token.weight
assert model20.output.weight.data_ptr() == model20.tgt_embed.token.weight.data_ptr()
assert not any(isinstance(m, (nn.MultiheadAttention, nn.Transformer)) for m in model20.modules())
print({"unique_trainable_parameters": parameter_count20})

## 7. Reverse toy：shifted target 与 teacher forcing

源序列是 3 个普通 token 加 EOS；目标是把 3 个 token 逆序后加 EOS。训练输入在最前放 BOS，并移除最后一个监督 token。这个数据集故意很小，目的只是检测 forward、mask、loss 和 decode 是否闭环。

`ignore_index=PAD` 让填充位置不进入损失；源/目标仍保留 EOS，防止解码只能依赖固定长度。

In [ ]:
def build_reverse_data():
    sequences = [
        [3, 4, 5], [3, 5, 6], [4, 6, 7], [5, 7, 8],
        [6, 8, 9], [7, 9, 10], [8, 10, 11], [9, 11, 12],
        [3, 7, 11], [4, 8, 12], [5, 9, 3], [6, 10, 4],
    ]
    src = torch.tensor([s + [EOS] for s in sequences])
    target = torch.tensor([list(reversed(s)) + [EOS] for s in sequences])
    tgt_in = torch.cat([torch.full((len(sequences), 1), BOS), target[:, :-1]], dim=1)
    return src, tgt_in, target

src20, tgt_in20, tgt_out20 = build_reverse_data()
assert src20.shape == tgt_in20.shape == tgt_out20.shape == (12, 4)
assert torch.equal(tgt_in20[:, 0], torch.full((12,), BOS))
assert torch.equal(tgt_out20[:, -1], torch.full((12,), EOS))
assert torch.equal(tgt_out20[0, :3], src20[0, :3].flip(0))

## 8. 属性测试：修改未来 token 不得改变过去 logits

在 `eval` 模式下构造两个目标前缀：位置 0、1 相同，从位置 2 开始不同。若 causal mask 正确，则前两个位置的 logits 应完全一致；若误用双向 self-attention，这一断言会失败。

In [ ]:
model20.eval()
prefix_a = torch.tensor([[BOS, 5, 6, 7]])
prefix_b = torch.tensor([[BOS, 5, 12, 11]])
with torch.no_grad():
    future_a = model20(src20[:1], prefix_a)
    future_b = model20(src20[:1], prefix_b)

assert torch.allclose(future_a[:, :2], future_b[:, :2], atol=1e-6)
assert not torch.allclose(future_a[:, 2:], future_b[:, 2:], atol=1e-5)
assert torch.isfinite(future_a).all()

## 9. 受控过拟合训练

使用全批量 AdamW 与 token 交叉熵。每次反向传播后检查梯度有限，并裁剪全局范数。验收看 token 准确率、精确序列匹配率和 loss 降幅；这些数值仅属于训练集合。

In [ ]:
torch.manual_seed(SEED + 1)
model20 = Seq2SeqTransformer(vocab_size=14, d_model=32, heads=4, d_ff=64, layers=1, max_len=8)
opt20 = torch.optim.AdamW(model20.parameters(), lr=0.015, weight_decay=0.0)
losses20 = []
model20.train()
for step in range(320):
    opt20.zero_grad(set_to_none=True)
    logits = model20(src20, tgt_in20)
    loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt_out20.reshape(-1), ignore_index=PAD)
    loss.backward()
    if step == 0:
        grad_tensors = [p.grad for p in model20.parameters() if p.grad is not None]
        assert grad_tensors and all(torch.isfinite(g).all() for g in grad_tensors)
        assert sum(float(g.abs().sum()) for g in grad_tensors) > 0
    torch.nn.utils.clip_grad_norm_(model20.parameters(), 1.0)
    opt20.step()
    losses20.append(float(loss.detach()))

model20.eval()
with torch.no_grad():
    trained_logits = model20(src20, tgt_in20)
    teacher_pred = trained_logits.argmax(-1)
token_acc20 = (teacher_pred == tgt_out20).float().mean().item()
exact_teacher20 = (teacher_pred == tgt_out20).all(1).float().mean().item()

assert losses20[-1] < losses20[0] * 0.15
assert token_acc20 >= 0.98
assert exact_teacher20 >= 0.90
assert torch.isfinite(trained_logits).all()
print({"loss_first": losses20[0], "loss_last": losses20[-1],
       "teacher_token_acc": token_acc20, "teacher_exact": exact_teacher20})

## 10. 贪心解码不是 teacher forcing

推理时没有真实前一 token。每步重新运行 decoder，取最后位置 argmax，并把已生成序列拼回输入；所有样本遇到 EOS 后结束。真实服务还需要最大长度、非法 token 过滤、beam cache 和批处理调度。

In [ ]:
@torch.no_grad()
def greedy_decode(model, src, max_new_tokens):
    model.eval()
    memory, src_mask = model.encode(src)
    generated = torch.full((src.shape[0], 1), BOS, dtype=torch.long, device=src.device)
    finished = torch.zeros(src.shape[0], dtype=torch.bool, device=src.device)
    for _ in range(max_new_tokens):
        next_token = model.decode(generated, memory, src_mask)[:, -1].argmax(-1)
        next_token = torch.where(finished, torch.full_like(next_token, EOS), next_token)
        generated = torch.cat([generated, next_token[:, None]], dim=1)
        finished |= next_token.eq(EOS)
        if bool(finished.all()):
            break
    return generated[:, 1:]

decoded20 = greedy_decode(model20, src20, max_new_tokens=4)
greedy_exact20 = (decoded20 == tgt_out20).all(1).float().mean().item()
assert decoded20.shape == tgt_out20.shape
assert greedy_exact20 >= 0.90
assert torch.equal(decoded20[:, -1], torch.full((12,), EOS))
print({"greedy_exact_on_controlled_set": greedy_exact20, "example": decoded20[0].tolist()})

## 11. 常见失败模式

- 只给 loss 加 `ignore_index`，却不在注意力中屏蔽 PAD：padding 仍可污染其他 token。
- causal 三角矩阵方向写反：训练指标异常好，但自回归解码崩溃。
- 把 `tgt_out` 原样喂给 decoder：当前位置直接看到答案，属于标签泄漏。
- 共享权重时先创建 optimizer、后替换 Parameter：optimizer 可能仍追踪旧权重。
- 只报告 teacher-forcing token accuracy：它不能替代真实 greedy/beam 序列指标。
- 无最大生成长度或 EOS 策略：坏模型会无限生成。

## 12. 制品与版本合同

manifest 必须包含特殊 token、最大长度、位置编码类型、归一化顺序、层数/头数、权重共享方式、词表哈希和 PyTorch 版本。仅有 `state_dict` 无法判断同一组矩阵应按哪种 mask 语义运行。

In [ ]:
manifest20 = {
    "artifact": "seq2seq_transformer_from_scratch",
    "schema_version": 1,
    "config": model20.config,
    "special_tokens": {"pad": PAD, "bos": BOS, "eos": EOS},
    "attention_mask": "bool_true_means_allowed",
    "norm": "post_norm",
    "target_contract": "BOS + target[:-1]",
    "weight_tying": "output.weight is tgt_embed.token.weight",
    "torch_version": torch.__version__,
}
buf20 = io.BytesIO()
torch.save({"manifest": manifest20, "state_dict": model20.state_dict()}, buf20)
artifact20 = buf20.getvalue()
sha20 = hashlib.sha256(artifact20).hexdigest()
buf20.seek(0)
loaded20 = torch.load(buf20, map_location="cpu", weights_only=False)
clone20 = Seq2SeqTransformer(**loaded20["manifest"]["config"])
clone20.load_state_dict(loaded20["state_dict"])
clone20.eval()
with torch.no_grad():
    clone_logits20 = clone20(src20, tgt_in20)

assert len(sha20) == 64
assert loaded20["manifest"]["special_tokens"]["bos"] == BOS
assert clone20.output.weight is clone20.tgt_embed.token.weight
assert torch.allclose(clone_logits20, trained_logits, atol=1e-7)
print({"sha256": sha20[:16] + "…", "bytes": len(artifact20)})

## 13. 生产替换与安全

教学实现用 Python 组合算子，便于理解，不保证 FlashAttention、KV cache、混合精度、分布式训练或 ONNX 导出性能。生产替换必须保留这里的 mask 属性测试与 decode 回归样例。

输入侧应限制 token 数、batch、生成长度与词表范围；按租户隔离 KV cache；记录截断率、EOS 到达率、生成长度、延迟分位数、无穷/NaN 计数和模型指纹。不要反序列化不可信 checkpoint。

## 14. 原始资料

- Vaswani et al., *Attention Is All You Need* (2017)：https://arxiv.org/abs/1706.03762
- PyTorch `nn.Module`：https://pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch broadcasting semantics：https://pytorch.org/docs/stable/notes/broadcasting.html
- PyTorch `CrossEntropyLoss`：https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html

架构和缩放点积公式来自原论文；这里的类、mask 约定和测试为独立教学实现。

In [ ]:
# 最终合同：非法 head、非法 mask、超长位置都必须失败。
def raises_value_error(fn):
    try:
        fn()
        return False
    except (ValueError, TypeError):
        return True

assert raises_value_error(lambda: MultiHeadAttentionFromScratch(10, 3))
assert raises_value_error(lambda: scaled_dot_product_attention(q0, k0, v0,
                              torch.zeros(1, 1, 1, 2, dtype=torch.bool)))
assert raises_value_error(lambda: emb0(torch.ones(1, 9, dtype=torch.long)))
assert model20.config["d_model"] % model20.config["heads"] == 0
assert manifest20["attention_mask"] == "bool_true_means_allowed"
assert losses20[-1] < 0.1
assert greedy_exact20 <= 1.0
assert sha20 == hashlib.sha256(artifact20).hexdigest()
assert all(torch.isfinite(p).all() for p in model20.parameters())
print("Notebook 20：全部合同测试通过。")